# 19 · Robustness analysis — noisy test signals and compressed time–frequency images

**(a) Clean vs noisy test signals** (24AIM301 Unit 1: noises). Additive noise is injected into
the preprocessed test recordings at SNR ∈ {20, 10, 5, 0} dB, three types:
white (sensor), pink 1/f (ambient / body), impulsive bursts (stethoscope friction).

**(b) Lossy image compression** (24AIM301 Unit 4). The STFT and wavelet-packet images fed to
the 2-D branches are coded with an 8 × 8 block-DCT + JPEG quantisation table at decreasing
quality. The Shannon entropy of the quantised symbols gives the achievable bits/pixel (first
theorem) and PSNR is the fidelity criterion. How far can the images be compressed before the
diagnosis degrades?

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import zlib
import matplotlib.pyplot as plt
import torch

device = torch.device("cuda")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
te = np.where(meta.split == "test")[0]
yte = meta.y.values[te]
fs = CFG["preprocessing"]["fs"]
proposed = load_json(P["results"] / "proposed_model.json")["proposed_model"]
fpol = load_json(P["results"] / "policies" / ("rl_joint_refined_s0.json" if proposed == "R4_full_rl_joint_refined" else "rl_full.json"))
base = config_dict(space, space.baseline)
MODELS = {"Raw-waveform CNN": ("B1_raw_cnn", ("T",), [dict(base, fusion="T")] * len(meta)),
          "Fixed STFT CNN": ("B2_stft_cnn", ("S",), [dict(base, fusion="S")] * len(meta)),
          "Fixed db6 wavelet CNN": ("B3_db6_wavelet_cnn", ("W",), [dict(base, fusion="W")] * len(meta)),
          "Fixed tri-branch fusion (no RL)": ("B5_fixed_TSW_fusion", ("T", "S", "W"), [base] * len(meta)),
          "Proposed RL fusion": (proposed, ("T", "S", "W"), policy_configs(meta, space, np.array(fpol["policy"]), fpol["K"]))}
loaded = {k: load_model(n, 0, CFG, P, branches=b).to(device) for k, (n, b, _) in MODELS.items()}
thr = {k: load_summary(n, 0, P)["threshold"] for k, (n, _, _) in MODELS.items()}

In [ ]:
rows = []
for kind in ("white", "pink", "impulse"):
    for snr in (None, 20, 10, 5, 0):
        if snr is None and kind != "white":
            continue
        sig_te = [signals[i] if snr is None else add_noise(signals[i], snr, np.random.default_rng(zlib.crc32(f"{meta.record_id[i]}{kind}".encode())), kind, fs) for i in te]
        for label, (name, branches, cfgs) in MODELS.items():
            st = RecordStore(sig_te, [cfgs[i] for i in te], fs, CFG["deep"])
            p = predict(loaded[label], st, np.arange(len(te)), device)
            m = binary_metrics(yte, p, thr[label])
            rows.append({"noise": kind, "snr_db": "clean" if snr is None else snr, "model": label, **m})
rob = pd.DataFrame(rows)
rob.to_csv(P["tables"] / "19_noise_robustness.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5), sharey=True)
order = ["clean", 20, 10, 5, 0]
for a, kind in zip(ax, ("white", "pink", "impulse")):
    for label in MODELS:
        d = pd.concat([rob[(rob.snr_db == "clean") & (rob.model == label)].iloc[:1], rob[(rob.noise == kind) & (rob.model == label) & (rob.snr_db != "clean")]])
        a.plot([str(s) for s in d.snr_db], d.macro_f1, "o-", label=label)
    a.set_title(f"{kind} noise"); a.set_xlabel("SNR (dB)")
ax[0].set_ylabel("test macro-F1"); ax[0].legend(fontsize=8)
plt.tight_layout(); savefig(fig, "19_noise_robustness"); plt.show()
rob.pivot_table(index=["noise", "snr_db"], columns="model", values="macro_f1", sort=False).round(3)

In [ ]:
crow = []
for q in (None, 90, 50, 25, 10, 5, 2):
    for label, (name, branches, cfgs) in MODELS.items():
        if branches == ("T",):
            continue
        stats = {}
        tf = None if q is None else (lambda img, q=q: jpeg_like(img, q, stats))
        st = RecordStore([signals[i] for i in te], [cfgs[i] for i in te], fs, CFG["deep"])
        p = predict(loaded[label], st, np.arange(len(te)), device, img_transform=tf)
        m = binary_metrics(yte, p, thr[label])
        crow.append({"quality": "none" if q is None else q, "model": label, "bits_per_pixel": stats.get("bits_per_pixel", 8.0),
                     "psnr_db": stats.get("psnr_db", np.inf), "macro_f1": m["macro_f1"], "auroc": m["auroc"]})
comp = pd.DataFrame(crow)
comp.to_csv(P["tables"] / "19_tf_image_compression.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for label in comp.model.unique():
    d = comp[comp.model == label]
    ax[0].plot(d.bits_per_pixel, d.macro_f1, "o-", label=label); ax[1].plot(d.psnr_db.replace(np.inf, 60), d.auroc, "o-", label=label)
ax[0].set_xscale("log"); ax[0].set_xlabel("entropy-coded bits / pixel (8 = uncompressed)"); ax[0].set_ylabel("test macro-F1"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("PSNR of TF image (dB; 60 = uncompressed)"); ax[1].set_ylabel("test AUROC")
plt.tight_layout(); savefig(fig, "19_compression"); plt.show()
comp.round(3)